# FSG Customer Service Agent

### Modular layout (build order)

| Block | Role |
|-------|------|
| **Catalog** | `get_product`, `find_similar_products`, `format_product_quote` — all read `fsg.db` |
| **Mockup** | `generate_fence_mockup_pil` — DALL·E 3, Dalí-style prompt |
| **Voice** | `communicate` — TTS mp3 bytes |
| **Tools** | `CHARLIE_TOOLS` — schemas for search, quote, mockup |
| **Agent loop** | `handle_charlie_tool_calls` → `chat` — mirrors day5 tool loop + audio + image |
| **UI** | `charlie_ui` — Gradio Blocks (chatbot + image + audio) |

Uncomment `charlie_ui.launch(...)` in the last cell after setting your API key.

In [13]:
import os
import json
import base64
import difflib
from io import BytesIO

import pandas as pd
import sqlite3
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "fsg.db"

In [15]:
system_message = """
You are **Chain Link Charlie**, the charismatic sales rep for Fencing Supply Group.
You have smooth small talk—warm, witty, never pushy. Keep chat replies to 1-2 sentences unless listing products.

**Conversation flow:**
1. Introduce yourself as Chain Link Charlie.
2. Greet and banter naturally until the customer mentions a fencing product or need.
3. If they name something vague or wrong, call **find_similar_products** with their words. Suggest a few
   **exact product_name** options from the tool results so they can pick one.
4. Once they confirm which product (by name or product_id), you may call **get_product_quote** to state price and stock accurately.
5. Ask where they plan to **install** the fence (city, property type, or setting). Wait until they answer.
6. Only after you have a confirmed **product_id** AND **install location**, call **create_fence_mockup** with both.
   Then summarize the quote aloud (from tool data)—the app will show a visual mockup separately.

Never invent prices or stock—always use tools for catalog data. If a tool returns no matches, apologize and ask them to rephrase.
"""

In [16]:
# Read in data and save to db

data = pd.read_csv("fencing_products_demo.csv")

with sqlite3.connect(DB) as conn:
    data.to_sql("products", conn, if_exists="replace", index=False)

In [17]:
data.head()

,product_id,product_name,category,material,height_ft,width_ft,color_or_finish,price_usd,stock_status,description
0,FSG-001,Cedar Privacy Fence Panel,Panel,Western Red Cedar,6.0,8.0,Natural Cedar,129.99,In Stock,Premium western red cedar privacy panel ideal ...
1,FSG-002,Cedar Dog-Ear Fence Board,Board,Western Red Cedar,6.0,0.5,Natural Cedar,4.25,In Stock,Traditional dog-ear cedar picket used for clas...
2,FSG-003,Pressure-Treated Pine Fence Board,Board,Pressure Treated Pine,6.0,0.5,Green Treated,3.10,In Stock,Affordable treated pine board designed for lon...
3,FSG-004,Black Aluminum Pool Fence Panel,Panel,Aluminum,5.0,6.0,Black Powder Coat,189.99,In Stock,Decorative aluminum panel commonly used for po...
4,FSG-005,Aluminum Fence Gate 48in,Gate,Aluminum,4.0,4.0,Black Powder Coat,249.00,In Stock,Pre-assembled aluminum gate compatible with al...


In [18]:
# --- Catalog (fsg.db) ---------------------------------------------------------

def get_product(product_id: str) -> dict | None:
    """Lookup by product_id (e.g. FSG-001)."""
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.execute("SELECT * FROM products WHERE product_id = ?", (product_id,))
        row = cur.fetchone()
    return dict(row) if row else None


# def find_similar_products(search_term: str, limit: int = 5) -> list[dict]:
#     """Return similar rows {product_id, product_name} from DB for fuzzy / partial names."""
#     term = (search_term or "").strip().lower()
#     with sqlite3.connect(DB) as conn:
#         rows = conn.execute("SELECT product_id, product_name FROM products").fetchall()
#     scored: list[tuple[float, str, str]] = []
#     for product_id, name in rows:
#         n = name.lower()
#         if not term:
#             scored.append((0.0, product_id, name))
#             continue
#         if n == term:
#             scored.append((100.0, product_id, name))
#         elif n.startswith(term):
#             scored.append((85.0, product_id, name))
#         elif term in n:
#             scored.append((70.0, product_id, name))
#         else:
#             ratio = difflib.SequenceMatcher(None, term, n).ratio()
#             if ratio >= 0.45:
#                 scored.append((ratio * 60.0, product_id, name))
#             qw, nw = set(term.split()), set(n.split())
#             if qw & nw:
#                 scored.append((35.0 + 10 * len(qw & nw), product_id, name))
#     scored.sort(key=lambda x: -x[0])
#     seen, out = set(), []
#     for _, pid, pname in scored:
#         if pid not in seen:
#             seen.add(pid)
#             out.append({"product_id": pid, "product_name": pname})
#         if len(out) >= limit:
#             break
#     if not out and term:
#         for pid, name in rows:
#             if any(w in name.lower() for w in term.split() if len(w) > 2):
#                 out.append({"product_id": pid, "product_name": name})
#             if len(out) >= limit:
#                 break
#     return out


def format_product_quote(p: dict) -> str:
    """Human-readable quote line for tools / Charlie."""
    return (
        f"{p['product_name']} ({p['product_id']}): ${float(p['price_usd']):.2f} USD, "
        f"{p['stock_status']}, {p['material']}, {p['height_ft']}ft x {p['width_ft']}ft, {p['color_or_finish']}."
    )

In [19]:
# --- Image mockup (DALL·E 3) ----------------------------------------------------

def generate_fence_mockup_pil(product_name: str, install_location: str) -> Image.Image:
    """Creative mockup: product installed at customer's described location (Dali-inspired)."""
    prompt = (
        f"A striking, surreal dreamlike scene: the fencing product \"{product_name}\" installed at "
        f"{install_location}. Show the fence clearly as the hero of the image. "
        f"Style influenced by Antoni Gaudi, vivid and imaginative as if hallucinatory, "
        f"but the product should still be recognizable. No text in the image."
    )
    r = openai.images.generate(
        model="dall-e-3",
        prompt=prompt,
        response_format="b64_json",
        size="1024x1024",
    )
    raw = base64.b64decode(r.data[0].b64_json)
    return Image.open(BytesIO(raw))

In [20]:
# --- Voice ----------------------------------------------------------------------

def communicate(message: str) -> bytes:
    """TTS for Charlie's spoken reply (mp3 bytes)."""
    response = openai.audio.speech.create(
        model="tts-1",
        voice="alloy",
        input=message,
    )
    return response.content

In [21]:
# --- OpenAI tool schemas --------------------------------------------------------

_find_similar_fn = {
    "name": "find_similar_products",
    "description": "Search the catalog when the customer names a product vaguely or incorrectly. Returns similar product_id and product_name rows from the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "search_term": {
                "type": "string",
                "description": "What the customer said (partial name, typo, or description).",
            }
        },
        "required": ["search_term"],
        "additionalProperties": False,
    },
}
_get_quote_fn = {
    "name": "get_product_quote",
    "description": "Get authoritative price, stock, and specs after the customer has picked a product. Use product_id (e.g. FSG-001).",
    "parameters": {
        "type": "object",
        "properties": {
            "product_id": {"type": "string", "description": "Catalog product_id."},
        },
        "required": ["product_id"],
        "additionalProperties": False,
    },
}
_mockup_fn = {
    "name": "create_fence_mockup",
    "description": "Call ONLY after product is confirmed AND the customer gave an install location. Generates a visual mockup and locks in the quote context.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_id": {"type": "string"},
            "install_location": {
                "type": "string",
                "description": "Where the fence goes, e.g. Miami backyard, rural Oregon property.",
            },
        },
        "required": ["product_id", "install_location"],
        "additionalProperties": False,
    },
}

CHARLIE_TOOLS = [
    {"type": "function", "function": _find_similar_fn},
    {"type": "function", "function": _get_quote_fn},
    {"type": "function", "function": _mockup_fn},
]

In [22]:
# --- Tool execution + chat loop (Gradio callback) -------------------------------

def handle_charlie_tool_calls(message) -> tuple[list, dict | None]:
    """
    Run DB/tools for each model tool call. Returns (tool_response_messages, mockup_job_or_none).
    mockup_job is set when create_fence_mockup succeeds—used to render DALL·E after the turn.
    """
    responses = []
    mockup_job = None
    for tool_call in message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments or "{}")
        if name == "find_similar_products":
            matches = find_similar_products(args.get("search_term", ""), limit=6)
            body = (
                json.dumps(matches, indent=2)
                if matches
                else "No close matches. Ask the customer to describe the fence type or material again."
            )
        elif name == "get_product_quote":
            p = get_product(args.get("product_id", ""))
            body = json.dumps(p, default=str) if p else "Unknown product_id. Use find_similar_products first."
        elif name == "create_fence_mockup":
            pid = args.get("product_id", "")
            loc = args.get("install_location", "").strip() or "unspecified location"
            p = get_product(pid)
            if p:
                mockup_job = {
                    "product_name": p["product_name"],
                    "install_location": loc,
                    "product_id": pid,
                }
                body = (
                    "Mockup queued for the UI. Official quote for your reply: "
                    + format_product_quote(p)
                    + " Remind them this is list price; installation extra."
                )
            else:
                body = "Invalid product_id. Do not claim a mockup was created."
        else:
            body = json.dumps({"error": f"Unknown tool: {name}"})
        responses.append(
            {"role": "tool", "content": body, "tool_call_id": tool_call.id}
        )
    return responses, mockup_job


def chat(history: list) -> tuple[list, bytes | None, Image.Image | None]:
    """
    One user turn: run Charlie with tools until final text, then TTS + optional fence mockup image.
    history is Gradio message list (last message should be user).
    """
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=CHARLIE_TOOLS
    )
    mockup_job = None
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_msgs, job = handle_charlie_tool_calls(msg)
        if job:
            mockup_job = job
        assistant_payload = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            assistant_payload["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]
        messages.append(assistant_payload)
        messages.extend(tool_msgs)
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=CHARLIE_TOOLS
        )
    reply = (response.choices[0].message.content or "").strip() or (
        "Hey—I'm Charlie. What kind of fence are we talking about today?"
    )
    history = history + [{"role": "assistant", "content": reply}]
    voice = communicate(reply)
    image = (
        generate_fence_mockup_pil(mockup_job["product_name"], mockup_job["install_location"])
        if mockup_job
        else None
    )
    return history, voice, image

In [23]:
# --- Gradio UI (same layout idea as day5) ---------------------------------------

def put_user_message_in_chat(message: str, history: list):
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks(title="Chain Link Charlie") as charlie_ui:
    gr.Markdown("## Fencing Supply Group — chat with **Chain Link Charlie**")
    with gr.Row():
        chatbot = gr.Chatbot(height=480, type="messages")
        image_out = gr.Image(height=480, label="Fence mockup", interactive=False)
    with gr.Row():
        audio_out = gr.Audio(label="Charlie", autoplay=True)
    with gr.Row():
        msg = gr.Textbox(label="Your message", placeholder="Ask about a fence, banter, or share your install location…")

    msg.submit(
        put_user_message_in_chat,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot],
    ).then(
        chat,
        inputs=[chatbot],
        outputs=[chatbot, audio_out, image_out],
    )

charlie_ui.launch(inbrowser=True, auth=("El Capitan", "fence"))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# --- TF-IDF product similarity (override) -------------------------------------
# Overrides the earlier heuristic version so the tool uses TF-IDF + cosine similarity.

_TFIDF_INDEX = None


def find_similar_products(search_term: str, limit: int = 5) -> list[dict]:
    """Return similar rows {product_id, product_name} using TF-IDF + cosine similarity."""
    term = (search_term or "").strip()
    if not term:
        return []

    global _TFIDF_INDEX
    if _TFIDF_INDEX is None:
        with sqlite3.connect(DB) as conn:
            rows = conn.execute("SELECT product_id, product_name FROM products").fetchall()

        names = [name for (_pid, name) in rows]
        vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
        )
        matrix = vectorizer.fit_transform(names)
        _TFIDF_INDEX = {"rows": rows, "vectorizer": vectorizer, "matrix": matrix}

    rows = _TFIDF_INDEX["rows"]
    vectorizer = _TFIDF_INDEX["vectorizer"]
    matrix = _TFIDF_INDEX["matrix"]

    query_vec = vectorizer.transform([term])
    scores = cosine_similarity(query_vec, matrix).flatten()

    top_idx = scores.argsort()[::-1][: max(limit, 1)]

    out: list[dict] = []
    for idx in top_idx:
        # Filter out absolute zeros to avoid irrelevant suggestions.
        if scores[idx] <= 0:
            continue
        product_id, product_name = rows[idx]
        out.append({"product_id": product_id, "product_name": product_name})
        if len(out) >= limit:
            break

    return out

Traceback (most recent call last):
  File "/Users/morgan/Desktop/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/morgan/Desktop/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/morgan/Desktop/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/morgan/Desktop/repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users